<br><h1>Fake News Detection Part 3</h1><br>

In [1]:
import polars as pl
import numpy as np

# Read in data
df = pl.read_csv("../news/data/995,000_rows_preprocessed.csv", columns=["type", "content"])
df.head()

type,content
str,str
"""political""","""plus one articl googl plus tha…"
"""fake""","""cost best senat bank committe …"
"""satire""","""man awoken <NUM> -year coma co…"
"""reliable""","""julia geist ask draw pictur co…"
"""conspiracy""","""<NUM> compil studi vaccin dang…"


In [2]:
# All types
with pl.Config(tbl_rows=-1):
    print(df["type"].value_counts().sort("count", descending=True))

shape: (14, 2)
┌────────────────────────────┬────────┐
│ type                       ┆ count  │
│ ---                        ┆ ---    │
│ str                        ┆ u32    │
╞════════════════════════════╪════════╡
│ reliable                   ┆ 218564 │
│ political                  ┆ 194518 │
│ bias                       ┆ 133232 │
│ fake                       ┆ 104883 │
│ conspiracy                 ┆ 97314  │
│ rumor                      ┆ 56445  │
│ nan                        ┆ 47786  │
│ unknown                    ┆ 43534  │
│ unreliable                 ┆ 35332  │
│ clickbait                  ┆ 27412  │
│ junksci                    ┆ 14040  │
│ satire                     ┆ 13160  │
│ hate                       ┆ 8779   │
│ 2018-02-10 13:43:39.521661 ┆ 1      │
└────────────────────────────┴────────┘


In [3]:
real_labels = ['reliable']
# fake_labels = "fake", "conspiracy", "rumor", "unreliable", "juncsci", "hate", "political", "satire", "bias"
drop_labels = ['unknown', 'nan', '2018-02-10 13:43:39.521661']
# This could effect real world use on data containing biased, satire etc. articles
# Also makes the training set smaller

df = df.filter(
    ~pl.col("type").is_in(drop_labels)).with_columns(
    pl.when(pl.col("type").is_in(real_labels))
    .then(pl.lit("real"))
    .otherwise(pl.lit("fake"))
    .alias("label")
)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer as _TV

n = df.shape[0]
indices = np.arange(n)
labels = df["label"].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(
    indices, test_size=0.2, random_state=1, stratify=labels
)
# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=1, stratify=labels[temp_idx]
)

df_train = df[train_idx.tolist()]
df_val = df[val_idx.tolist()]
df_test = df[test_idx.tolist()]

print(f"Train: {df_train.shape[0]}  ({df_train.shape[0] / n:.0%})")
print(f"Val:   {df_val.shape[0]}  ({df_val.shape[0] / n:.0%})")
print(f"Test:  {df_test.shape[0]}  ({df_test.shape[0] / n:.0%})")


Train: 722943  (80%)
Val:   90368  (10%)
Test:  90368  (10%)


In [5]:
# Verify that stratified splitting preserved the label ratio
for name, split in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    dist = (
        split.group_by("label")
        .agg(pl.len().alias("count"))
        .sort("label")
        .with_columns(
            (pl.col("count").cast(pl.Float64) / pl.col("count").sum() * 100)
            .round(1)
            .alias("pct")
        )
    )
    print(f"\n{name} split:")
    print(dist)


Train split:
shape: (2, 3)
┌───────┬────────┬──────┐
│ label ┆ count  ┆ pct  │
│ ---   ┆ ---    ┆ ---  │
│ str   ┆ u32    ┆ f64  │
╞═══════╪════════╪══════╡
│ fake  ┆ 548092 ┆ 75.8 │
│ real  ┆ 174851 ┆ 24.2 │
└───────┴────────┴──────┘

Val split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 68512 ┆ 75.8 │
│ real  ┆ 21856 ┆ 24.2 │
└───────┴───────┴──────┘

Test split:
shape: (2, 3)
┌───────┬───────┬──────┐
│ label ┆ count ┆ pct  │
│ ---   ┆ ---   ┆ ---  │
│ str   ┆ u32   ┆ f64  │
╞═══════╪═══════╪══════╡
│ fake  ┆ 68511 ┆ 75.8 │
│ real  ┆ 21857 ┆ 24.2 │
└───────┴───────┴──────┘


In [6]:
import gc
from scipy.sparse import vstack
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=100_000, 
    ngram_range=(1, 2),  
    sublinear_tf=True,  
    min_df=3,
    max_df=0.95, 
    dtype=np.float32 
)

# Fit the vocabulary on just 30% of the training data
print("Fitting Vectorizer (30% sample)")
sample_series = df_train["content"].sample(fraction=0.3, seed=1)
vectorizer.fit(sample_series)
del sample_series; gc.collect()

# Helper function to transform in small bites
def chunked_transform(series, chunk_size=50_000):
    chunks = []
    n_rows = len(series)
    
    for start in range(0, n_rows, chunk_size):
        # Polars slicing
        length = min(chunk_size, n_rows - start)
        chunk_series = series.slice(start, length)
        
        chunk_matrix = vectorizer.transform(chunk_series)
        chunks.append(chunk_matrix)
        
    # Join the chunks into one final matrix
    return vstack(chunks, format="csr")

# Extract labels before we start deleting things
y_train = (df_train["label"] == "fake").cast(pl.Int8).to_numpy()
y_val   = (df_val["label"]   == "fake").cast(pl.Int8).to_numpy()
y_test  = (df_test["label"]  == "fake").cast(pl.Int8).to_numpy()
val_types = df_val["type"].to_numpy()
val_texts = df_val["content"].to_list()

# Vectorize train, then drop the df
print("Transforming Train")
tfidf_train = chunked_transform(df_train["content"])
del df_train; gc.collect()
print("Transforming Val")
tfidf_val = chunked_transform(df_val["content"])
del df_val; gc.collect()
print("Transforming Test")
tfidf_test = chunked_transform(df_test["content"])
del df_test; gc.collect()

# Also drop the full df if still around (Let the memory be freed!!)
try:
    del df; gc.collect()
except NameError:
    pass

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"TF-IDF train: {tfidf_train.shape}")

Fitting Vectorizer (30% sample)
Transforming Train
Transforming Val
Transforming Test
Vocabulary size: 100,000
TF-IDF train: (722943, 100000)


In [7]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

# Try a logarithmic sweep of C values and pick the best one on the validation set
# c_values = np.logspace(-4, 1, 8)
# tuning_rows = []
# best_c = None
# best_f1 = -1.0

# for c_value in c_values:
#     candidate = LinearSVC(C=float(c_value), max_iter=10_000, random_state=1)
#     candidate.fit(tfidf_train, y_train)

#     y_val_candidate = candidate.predict(tfidf_val)
#     val_accuracy = accuracy_score(y_val, y_val_candidate)
#     val_f1 = f1_score(y_val, y_val_candidate)

#     tuning_rows.append({
#         "C": float(c_value),
#         "val_accuracy": val_accuracy,
#         "val_f1": val_f1,
#     })
#     print(f"{c_value} done")
#     if val_f1 > best_f1:
#         best_f1 = val_f1
#         best_c = float(c_value)

# tuning_results = (
#     pl.DataFrame(tuning_rows)
#     .sort(["val_f1", "val_accuracy"], descending=[True, True])
# )
# print(tuning_results)

# Best C
best_c = 0.3727593720314942

# Refit one final model using the selected C
# svm = LinearSVC(C=best_c, max_iter=10_000, random_state=1, class_weight="balanced")
# svm.fit(tfidf_train, y_train)

# y_val_pred = svm.predict(tfidf_val)
# print(f"\nBest C: {best_c}")
# print("Validation:")
# print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
# print(f"F1: {f1_score(y_val, y_val_pred):.4f}")
# print(classification_report(y_val, y_val_pred, target_names=["real", "fake"]))

In [10]:
import os
import joblib
import pandas as pd
from scipy.sparse import save_npz

os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Save SVM model + TF-IDF vectorizer (used in part 4)
# joblib.dump(svm, "models/SVend-Martin.pkl")
# joblib.dump(vectorizer, "models/SVend-Martin_vect.pkl")

# Save SVM test matrix + matching labels (used in part 4)
save_npz("data/X_test_tfidf.npz", tfidf_test)
pd.DataFrame({"isfake": y_test}).to_csv("data/y_test_tfidf.csv", index=False)


In [9]:
import re
import joblib
from nltk.corpus import stopwords
import Stemmer

svm = joblib.load('training_results/SVend-Martin.pkl')
vectorizer = joblib.load('training_results/SVend-Martin_vect.pkl')


# Uncompiled patterns for polars later
email_str = r'[^ \n,"]+@[^ \n,"]+\.[^ \n,"]+'
date_str = r'[0-9]{2,4}[-/][0-9]{2,4}[-/][0-9]{2,4}'
url_str = r'(?:http)?s?(?://)?[^ \n,"]+\.[a-z]{2,}[^ \n,"]+'
num_str = r'[0-9]+[,.]?[0-9]*'
special_str = r'[^a-zA-Z\-<>\'’‘ʼ′＇]'
apostrephe_str = r'\'’‘ʼ′＇'
ws_str = r'\s+'

# Pattern strings — shared with Task 2 (Polars uses these directly)
email_re = re.compile(email_str)
date_re = re.compile(date_str)
url_re = re.compile(url_str)
num_re = re.compile(num_str)
special_re = re.compile(special_str)
apostrephe_re = re.compile(apostrephe_str)
ws_re = re.compile(ws_str)

# Initialize NLTK Stop words and PyStemmer
stop_words = set(stopwords.words("english"))
stemmer = Stemmer.Stemmer("english")

def preprocess_text(doc: str):
    # Cleaning
    lower_case = doc.lower()
    substituted = email_re.sub(" <EMAIL> ", lower_case)
    substituted = date_re.sub(" <DATE> ", substituted)
    substituted = url_re.sub(" <URL> ", substituted)
    substituted = num_re.sub(" <NUM> ", substituted)
    no_specials = special_re.sub(" ", substituted)
    no_apostrophes = apostrephe_re.sub("", no_specials)
    cleaned = ws_re.sub(' ', no_apostrophes).strip()

    # Stemming and Filtering
    words = [w for w in cleaned.split() if w not in stop_words]
    return " ".join(stemmer.stemWords(words))



def qualitative_test_from_file(filepath):
    # Read the file
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Split by the '---' delimiter, strip whitespace, and ignore empty chunks
    raw_articles = [chunk.strip() for chunk in content.split('---') if chunk.strip()]
    
    # Process text
    processed_texts = [preprocess_text(t) for t in raw_articles]
    
    # Vectorize the mapped texts
    X_custom = vectorizer.transform(processed_texts)
    
    # Get predictions
    # 1 means "fake" and 0 means "real"
    preds = svm.predict(X_custom)
    
    # Score the predictions (distance to hyperplane)
    scores = svm.decision_function(X_custom)
    source_names = [
        "Normal Reuters",
        "Semi-political Reuters",
        "Very-political Reuters",
        "Non-political The New Yorker",
        "Satire The Onion",
        "Fake-news and political InfoWars",
    ]

    for i, text in enumerate(raw_articles):
        print(f"{source_names[i]}")

        label = "FAKE" if preds[i] == 1 else "REAL"

        display_text = text[:80].replace("\n", " ") + "..."
        processed_display = processed_texts[i][:80] + "..."

        print("Raw Text:          ", display_text)
        print("Processed Text:    ", processed_display)
        print("Prediction (SVM):  ", label)
        print(f"SVM Score:         {scores[i]:.4f}")
        print("-" * 80)


qualitative_test_from_file('test_articles.txt')


FileNotFoundError: [Errno 2] No such file or directory: 'training_results/SVend-Martin.pkl'

In [ ]:
saved_svm = joblib.load('training_results/SVend-Martin.joblib')
saved_vectorizer = joblib.load('training_results/SVend-Martin_vect.joblib')

saved_tfidf_val = saved_vectorizer.transform(val_texts)
saved_y_val_pred = saved_svm.predict(saved_tfidf_val)
saved_y_val_score = saved_svm.decision_function(saved_tfidf_val)

saved_val_type_metrics = (
    pl.DataFrame({
        "type": val_types,
        "guess": saved_y_val_pred,
        "score": saved_y_val_score,
    })
    .group_by("type")
    .agg(
        pl.len().alias("count"),
        pl.col("guess").mean().round(3).alias("mean_guess"),
        pl.col("score").mean().round(3).alias("mean_score"),
    )
    .sort("count", descending=True)
)

print("Saved SVend-Martin on validation:")
print(f"Accuracy: {accuracy_score(y_val, saved_y_val_pred):.4f}")
print(f"F1: {f1_score(y_val, saved_y_val_pred):.4f}")
print(classification_report(y_val, saved_y_val_pred, target_names=["real", "fake"]))
print("Validation by article type:")
with pl.Config(tbl_rows=-1):
    print(saved_val_type_metrics)


Saved SVend-Martin on validation:
Accuracy: 0.9695
F1: 0.9798
              precision    recall  f1-score   support

        real       0.93      0.95      0.94     21856
        fake       0.98      0.98      0.98     68512

    accuracy                           0.97     90368
   macro avg       0.96      0.96      0.96     90368
weighted avg       0.97      0.97      0.97     90368

Validation by article type:
shape: (11, 4)
┌────────────┬───────┬────────────┬────────────┐
│ type       ┆ count ┆ mean_guess ┆ mean_score │
│ ---        ┆ ---   ┆ ---        ┆ ---        │
│ str        ┆ u32   ┆ f64        ┆ f64        │
╞════════════╪═══════╪════════════╪════════════╡
│ reliable   ┆ 21856 ┆ 0.054      ┆ -2.067     │
│ political  ┆ 19410 ┆ 0.965      ┆ 1.207      │
│ bias       ┆ 13267 ┆ 0.985      ┆ 1.573      │
│ fake       ┆ 10491 ┆ 0.982      ┆ 1.941      │
│ conspiracy ┆ 9682  ┆ 0.988      ┆ 1.444      │
│ rumor      ┆ 5708  ┆ 0.97       ┆ 1.289      │
│ unreliable ┆ 3645  ┆ 0.99  